# Decoding analysis
This notebook demonstrates a machine learning approach to analyse the data. The goal is to predict the stimulus from the neural activity. This information based approach is different from the traditional hypothesis testing approach. In the hypothesis testing approach, we test whether the neural activity is significantly different between different two conditions. In the information based approach, we test whether the neural activity contains information about the condition.


If you are interested in reading some neuroscience papers that uses decoding analysis, here are some examples:
- [Decoding Brain Responses to Names and Voices across Different Vigilance States](https://www.mdpi.com/1424-8220/21/10/3393)
- [Resolving human object recognition in space and time](https://www.nature.com/articles/nn.3635)
- [Neural dynamics of phoneme sequences reveal position-invariant code for content and order](https://www.nature.com/articles/s41467-022-34326-1)
- [Long-Term Variability in Visual Processing versus Perceptual Stability](https://www.eneuro.org/content/13/6/ENEURO.0344-25.2026)

More methods oriented papers:
- [Characterizing the dynamics of mental representations: the temporal generalization method](https://www.sciencedirect.com/science/article/abs/pii/S1364661314000199)
- [Information-based functional brain mapping](https://www.pnas.org/doi/10.1073/pnas.0600244103) (mainly for fMRI, but the idea is the same)
- [Deconstructing multivariate decoding for the study of brain function](https://pubmed.ncbi.nlm.nih.gov/28782682/)

and at last, a technical review paper discussing interpretations both of activation and information based analysis. More specifically they argue that most analysis focus on the information we can measure as experimenters (experimenter-as-receiver), rather than on how the rest of the brain interpret those activations (cortex-as-receiver). Can be helpful in terms of interpreting and discussing results:
- [Does neuroimaging measure information in the brain?](https://pubmed.ncbi.nlm.nih.gov/26833316/)

In [ ]:
import mne
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from tqdm import tqdm
import pickle as pkl


from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# importing from our local utils
import sys
sys.path.append("../scripts/")
from params import fnames

## Loading and preparing data
Just so you have something to play around with, I've added some epochs and source time course to the `data_for_notebooks` folder here on uCloud. Note that the source time courses have been morphed to fsaverage!

In [ ]:
epochs_path = Path("/work/data_for_notebooks/0193_filtered_0.1-40_fir_ICA_meg-epo.fif")
epochs = mne.read_epochs(epochs_path)
epochs = epochs.drop_channels(epochs.info["bads"])
epochs.pick("mag") # for speed only use magnetometers in this notebook

To illustrate how to decode using MEG, I'll extract the epochs for gabor patches and mask respectively. Here I'm just using the magnetometers for speed. 

In [ ]:
cond_1 = "mask"
cond_2 = "stimulus"

stim_1_epochs = epochs[cond_1]
stim_2_epochs = epochs[cond_2]
stim_1 = stim_1_epochs.get_data()
stim_2 = stim_2_epochs.get_data()

print(stim_1.shape, stim_2.shape) # trials, sensors, timepoints

## Decoding within subject in sensor space
This notebook focuses on within-subject decoding. That is we train a model on data from one subject and test it on the same subject. However, we should always use a separate dataset for training and testing to avoid overfitting. This means that we cannot train and test the model on data from the same trials from our subject. Therefore, we use k-fold cross-validation. If you are not familiar with cross-validation, you can read more about it [here](https://spotintelligence.com/2023/07/29/k-fold-cross-validation/).


We need to make a variable that contains the labels for the two conditions `y`. We will use 0 for one condition and 1 for the other condition. Furthermore we concatenate the data from the two conditions into one variable `X`.

In [ ]:
y = np.array([0] * len(stim_1) + [1] * len(stim_2))

# concatenate the data along the trial axis
X = np.concatenate([stim_1, stim_2], axis=0)

print(X.shape)
print(y.shape)

Now that we have prepared the data and the labels, we can start the decoding analysis. We will use the scikit-learn library for this purpose. This library contains a lot of useful tools for machine learning.

First we will be using a simple decoding analysis, where we train the model on data from a single time point. Then we will use a more advanced decoding analysis, where we test the decoding accuracy at all time points. Finally, we will conduct a temporal generalization analysis, where we train the model on data from one time point and test it on all time points.

### Decoding at one time point
Lets begin by focusing on one time point. Here I use sample 95 as an example.

In [ ]:
# get data from a specific sample
sample = 95
X_t1 = X[:, :, sample]

<div class="alert alert-block alert-info">
<b> Question:</b> What time point before/after the mask/stimulus does sample 95 correspond to?

💡 <b>Hint:</b> the epochs.times attribute may help you figure it out!
</div>

Now lets try and plot the averaged data in the two conditions for the given time point. 

In [ ]:
evoked_stim_1 = stim_1_epochs.average()
evoked_stim_2 = stim_2_epochs.average()

all_data = np.concatenate([
    evoked_stim_1.data[:, sample],
    evoked_stim_2.data[:, sample]
])

vmax = np.max(np.abs(all_data)) * 1e15
vmin = -vmax

fig, axes = plt.subplots(
    1, 3,
    dpi=300,
    width_ratios=[1, 1, 0.1],
    gridspec_kw={"wspace": 0.3}
)

for cond, evoked, ax in zip([cond_1, cond_2],[evoked_stim_1, evoked_stim_2], axes[:2]):
    mne.viz.plot_evoked_topomap(
        evoked,
        axes=ax,
        times=epochs.times[sample],
        show=False,
        colorbar=False,
        vlim=(vmin, vmax),
    )
    
    ax.set_title(cond)


# COLOR BAR
# Get the image from the first topomap
im = axes[0].images[0]

# Shrink the colorbar axes vertically
pos = axes[2].get_position()

shrink = 0.4
new_height = pos.height * shrink
new_y = pos.y0 + (pos.height - new_height) / 2

axes[2].set_position([
    pos.x0,
    new_y,
    pos.width,
    new_height
])

# Add shared colorbar
cbar = fig.colorbar(im, cax=axes[2])
cbar.set_label(f"Amplitude")

plt.show()

<div class="alert alert-block alert-info">
<b> Question:</b> Do you think its gonna be a easy task for the decoder to classify? If yes, why? If no, why not?
</div>

Now lets set up some useful scikit learn classes which we will be using throughout the notebook! 


In [ ]:
ncv=10
scaler = StandardScaler()
kfold = StratifiedKFold(ncv, shuffle=True)

clf = LogisticRegression(
    random_state=99,
    C=1.0,
    max_iter=1000
)

# You can also play around with other classifiers instead!

#clf = SVC(
#    kernel="linear",
#    random_state=99,
#    C=1.0,
#    max_iter=1000
#)


#clf = LinearDiscriminantAnalysis()



In [ ]:
pipeline = Pipeline(
    [("scaler", scaler), ("clf", clf)]
)

scores = cross_val_score(
    pipeline, 
    X_t1, y, 
    cv=kfold
)
print(f"Mean accuracy = {np.mean(scores)}")

### Decoding across time points
Now we have tested the ability to decode the condition at one time point. However, it might be interesting to see how the decoding accuracy changes over the duration of the trial. We can do this by looping through all the time points. We will store the decoding accuracy for each time point.

This allows us to plot the decoding accuracy as a function of time, which informs when the condition can be decoded from the MEG signal.

In [ ]:
scores = np.empty((ncv, X.shape[-1]))

for t in tqdm(range(X.shape[-1])): # loop over time points
    X_t = X[:, :, t]
    scores_t = cross_val_score(
        pipeline, 
        X_t, y, 
        cv=kfold
    )
    scores[:, t] = scores_t

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6), dpi=300)

# Plot accuracy of each cross-validation fold
for score_cv in scores:
    ax.plot(
        epochs.times,
        score_cv*100,
        alpha=0.1,
        linewidth=1,
        color="darkgreen"
    )

# Plot the average across cross-validation folds
average_score = scores.mean(axis=0)

ax.plot(
    epochs.times,
    average_score*100,
    linewidth=2,
    color="darkgreen",
    label="Decoding accuracy\nacross CV folds"
)

# Aesthetics
ax.set_ylabel("Accuracy (%)")
ax.set_xlabel("Time (s)")
ax.axvline(0, color = "grey", linestyle = "--", label="Stimulus presentation")


# remove top and right spines
for spine_label in ["top", "right"]:
    ax.spines[spine_label].set_visible(False)

# add legend
ax.legend(frameon=False)

plt.tight_layout()
plt.show()

### Temporal generalisation
We can also use the model trained at one time point to predict the condition at another time point. This is called temporal generalisation. We can do this by looping through all the time points and training the model at one time point and testing it at all other time points.

In [ ]:
accuracies = np.zeros((ncv, X.shape[2], X.shape[2]))

# loop over the splits
for i, (train_inds, test_inds) in enumerate(kfold.split(X, y)):
    # get the train and test data
    X_train = X[train_inds]
    X_test = X[test_inds]

    y_train = y[train_inds]
    y_test = y[test_inds]

    # loop over the timesamples for training
    for t in range(X_train.shape[2]):
        
        # scale the data (notice that we fit the scaler to the training data and not the test data)
        X_train_scaled = scaler.fit_transform(X_train[:, :, t])

        # fit the classifier
        clf.fit(X_train_scaled, y_train)

        # loop over the timesamples for testing
        for tt in range(X_test.shape[2]):
            X_test_scaled = scaler.transform(X_test[:, :, tt])


            # get the accuracy
            accuracy = clf.score(X_test_scaled, y_test)

            # save the accuracy
            accuracies[i, t, tt] = accuracy

print(f"mean accuracy: {np.mean(accuracies)}")

In [ ]:
# make a shape (n_splits, n_timesamples, n_timesamples) array to store the accuracies
accuracies = np.zeros((ncv, X.shape[2], X.shape[2]))

# loop over the splits
for i, (train_inds, test_inds) in enumerate(kfold.split(X, y)):
    # get the train and test data
    X_train = X[train_inds]
    X_test = X[test_inds]

    y_train = y[train_inds]
    y_test = y[test_inds]

    # loop over the timesamples for training
    for t in tqdm(range(X_train.shape[2]), desc=f"split {i+1}/{ncv}, sample"):
        
        # scale the data (notice that we fit the scaler to the training data and not the test data)
        X_train_scaled = scaler.fit_transform(X_train[:, :, t])

        # fit the classifier
        clf.fit(X_train_scaled, y_train)

        # loop over the timesamples for testing
        for tt in range(X_test.shape[2]):
            X_test_scaled = scaler.transform(X_test[:, :, tt])


            # get the accuracy
            accuracy = clf.score(X_test_scaled, y_test)

            # save the accuracy
            accuracies[i, t, tt] = accuracy

print(f"mean accuracy: {np.mean(accuracies)}")

<div class="alert alert-block alert-info">
<b> Question:</b> We see now that the average accuracy all of a sudden is around 50%. Why might this be the case? Try to answer this question <b> before</b> plotting the results below!
</div>

In [ ]:
# take the mean over the splits
accuracies_avg = np.mean(accuracies, axis=0)

fig, ax = plt.subplots(1, figsize=(10, 10), dpi=300)

# plot the accuracies
im = ax.imshow(accuracies_avg, vmin=0, vmax=1, origin="lower", cmap="RdBu_r", interpolation="gaussian")
times = epochs.times

# only plot every 25th timepoint
ax.set_xticks(np.arange(len(times))[::25])
ax.set_xticklabels(times[::25])
ax.set_yticks(np.arange(len(times))[::25])
ax.set_yticklabels(times[::25])

# label the axes
ax.set_xlabel("Test time relative to onset (ms)")
ax.set_ylabel("Train time relative to onset (ms)")

ax.set_title("Temporal generalisation")

# add a colorbar
cbar = fig.colorbar(im, ax=ax, shrink=0.5)
cbar.set_label("Accuracy")

# plot lines at 0 indicating onset
ax.axhline(50, color="black", linestyle="--", linewidth=1)
ax.axvline(50, color="black", linestyle="--", linewidth=1)

<div class="alert alert-block alert-info">
<b> Question:</b> What might be the reason we are seeing this ripple pattern? Have a good think about the experimental design!
</div>

# Decoding in source space
I've included some source time courses from all the target presentations of one subject!

In [ ]:
subject = "0192"

stcs_path = Path(f"/work/data_for_notebooks/{subject}_morphed_to_fsaverage_filtered_0.1-40_fir_ICA_MNE.pkl")
with open(stcs_path, "rb") as f:
    stcs = pkl.load(f)

# loading the corresponding epochs and getting the target trials so we have the 
epochs_path = Path(f"/work/data_for_notebooks/{subject}_filtered_0.1-40_fir_ICA_meg-epo.fif")
epochs = mne.read_epochs(epochs_path, preload=False)
epochs.metadata["subjective_combined"] = np.where(
    epochs.metadata["subjective_response"].isin([1, 2]),
    0,
    1
)

stim_epochs = epochs["stimulus"]
stim_epochs.metadata["index"] = range(len(stim_epochs))


In [ ]:
metadata = stim_epochs.metadata

counts = metadata["subjective_combined"].value_counts()
print(counts)

# remove random trials from the subjective response with the largest number of trials
n = counts.min()

# Randomly select n trials from each response
metadata = metadata.groupby("subjective_combined").sample(
    n=n,
    random_state=99
)

# Get the epoch indices
idx = metadata["index"]

stcs = [stcs[i] for i in idx]
y =  metadata["subjective_combined"]


In [ ]:
len(stcs), len(metadata)

In [ ]:
## READING LABELS FROM ANNOTATION
labels = mne.read_labels_from_annot('fsaverage', subjects_dir=fnames.subjects_dir)

def extract_label(label, stcs):
    label_stcs = []
    for stc in stcs:
        label_stcs.append(stc.in_label(label))
        
    return label_stcs

label_stcs = extract_label(labels[1], stcs) 

In [ ]:
label_idx = 0

n_trials = len(y) ## how many repetitions
n_samples = label_stcs[0].data.shape[1] # how many time points

label_stcs = extract_label(labels[label_idx], stcs) 
    
n_vertices = label_stcs[0].data.shape[0]  # how many source positions in the label

X_source_space = np.empty((n_trials, n_vertices, n_samples))
    
for i, stc in enumerate(label_stcs):
    X_source_space[i] = stc.data

scores_source_space = np.empty((ncv, X_source_space.shape[-1]))

for t in tqdm(range(X_source_space.shape[-1])): # loop over time points
    X_t = X_source_space[:, :, t]
    scores_t = cross_val_score(pipeline, X_t, y, cv=kfold)
    scores_source_space[:, t] = scores_t

fig, ax = plt.subplots(1, 1, figsize=(10, 6), dpi=300)

# Plot accuracy of each cross-validation fold
for score_cv in scores_source_space:
    ax.plot(
        epochs.times,
        score_cv*100,
        alpha=0.1,
        linewidth=1,
        color="darkgreen"
    )


# Plot the average across cross-validation folds
average_score = scores_source_space.mean(axis=0)

ax.plot(
    epochs.times,
    average_score*100,
    linewidth=2,
    color="darkgreen",
    label="Decoding accuracy\nacross CV folds"
)

# Aesthetics
ax.set_ylabel("Accuracy (%)")
ax.set_xlabel("Time (s)")
ax.axvline(0, color = "grey", linestyle = "--", label="Stimulus presentation")


# remove top and right spines
for spine_label in ["top", "right"]:
    ax.spines[spine_label].set_visible(False)

# add legend
ax.legend(frameon=False)

ax.set_title(f"{labels[label_idx].name}, number of vertices = {n_vertices}")

plt.tight_layout()
plt.show()

## Challenge: Whole-brain decoding in source space

Instead of decoding using all vertices within each label, calculate the average activity across vertices within each label.

Create a new feature matrix $X$ where:

- Each **row** corresponds to a trial
- Each **column** corresponds to a label

Create $X$ for one time point and use it as input to the classifier.